# Sliding Window

---

## What is Sliding Window?

Sliding window is a technique where you maintain a **window** (a subarray or substring) that moves through the data from left to right. Instead of recomputing everything from scratch for each position, you **slide** the window — adding one element on the right and removing one on the left.

```
array:  [2, 1, 5, 1, 3, 2]   window size = 3

step 1: [2, 1, 5] 1  3  2    sum = 8
step 2:  2 [1, 5, 1] 3  2    sum = 7   (removed 2, added 1)
step 3:  2  1 [5, 1, 3] 2    sum = 9   (removed 1, added 3)
step 4:  2  1  5 [1, 3, 2]   sum = 6   (removed 5, added 2)
```

Each step is O(1) — just one addition and one subtraction — instead of recomputing the sum from scratch each time.

### Two types of sliding window

**Fixed size** — window size k is given. Move one step at a time.

**Variable size** — window expands and shrinks based on a condition. Used for problems like "longest substring without repeating characters".

---
## Part 1 — Fixed Size Window

**Pattern:**
1. Compute the sum of the first window (indices 0 to k-1)
2. Slide: add `nums[i]`, remove `nums[i-k]`
3. Track max/min/condition at each position

```
window_sum = window_sum + nums[i] - nums[i - k]
```

This is O(n) total — each element is added once and removed once.

In [1]:
# find the maximum sum of any subarray of size k
# example: nums=[2,1,5,1,3,2], k=3 → 9  (subarray [5,1,3])
#
# step 1: compute sum of first window (indices 0 to k-1)
# step 2: slide the window — add nums[i], remove nums[i-k]
# step 3: update max at each step

def max_sum_subarray(nums, k):
    window_sum = sum(nums[0:k])
    max_sum = window_sum

    for i in range(k, len(nums)):
        window_sum = window_sum + nums[i] - nums[i-k]
        if window_sum > max_sum:
            max_sum = window_sum
    return max_sum

print(max_sum_subarray([2, 1, 5, 1, 3, 2], 3))  # 9
print(max_sum_subarray([1, 4, 2, 9, 7, 3], 2))  # 16

9
16



### The formula
window_sum = window_sum + nums[i] - nums[i - k]

```python

- `nums[i]`     → new element entering from the right
- `nums[i - k]` → old element leaving from the left
- `i`           → index of the new element (loop variable)
- `k`           → window size

---

### How to implement it

```python
# step 1: build first window manually — before the loop
window_sum = sum(nums[0:k])
max_sum = window_sum

# step 2: slide from index k to end
for i in range(k, len(nums)):
    window_sum = window_sum + nums[i] - nums[i - k]
    max_sum = max(max_sum, window_sum)
```

---
## Part 2 — Variable Size Window

When window size isn't fixed, you use two pointers: `left` and `right`.

- `right` expands the window by moving forward each iteration
- `left` shrinks the window when a condition is violated

```
while condition violated:
    remove nums[left] from window
    left += 1
```

**The key insight:** instead of checking every possible subarray (O(n²)), you let the window grow until it breaks a rule, then shrink from the left until it's valid again. Each element is visited at most twice — once by `right`, once by `left` — so the total is O(n).

### Why a hash set?

For problems involving uniqueness (like "no repeating characters"), you need to know instantly:
- is this character already in the window? → O(1) with a set
- when shrinking, remove the leftmost character → O(1) with a set

A list would make both operations O(n).

In [6]:
# find the length of the longest subarray with all unique elements
# example: [1,2,3,1,2] → 3  (subarray [1,2,3] or [2,3,1])
#
# use left and right pointers + a set to track what's in the window
# right expands every iteration
# when nums[right] is already in the set → shrink from left until it's removed
# track max window size at each step

def longest_unique_subarray(nums):
    left = 0  # left pointer
    # right pointer is our loop variable i

    track_set = set()
    max_window = 0

    for i in range(len(nums)):
        if nums[i] not in track_set:
            track_set.add(nums[i])
        else:
            # shrink from left until the duplicate is removed
            while nums[i] in track_set:
                track_set.remove(nums[left])
                left += 1
            track_set.add(nums[i])
        max_window = max(max_window, len(track_set))

    return max_window   

print(longest_unique_subarray([1, 2, 3, 1, 2]))  # 3
print(longest_unique_subarray([1, 1, 1, 1]))      # 1
print(longest_unique_subarray([1, 2, 3, 4, 5]))   # 5

3
1
5


## Why Set Over List for Sliding Window

When tracking window contents, you need two operations every step:
1. **Check** — is this element already in the window?
2. **Remove** — remove the leftmost element when shrinking

---

### Speed comparison

| Operation | List | Set |
|-----------|------|-----|
| `x in window` | O(n) — scans every element | O(1) — hash lookup |
| `window.remove(x)` | O(n) — finds element first | O(1) — hash lookup |

---

### Impact on overall complexity

```python
# with list
for i in range(n):           # O(n)
    x in window_list         # O(n) ← kills performance
    window_list.remove(x)    # O(n) ← kills performance
# total: O(n²)

# with set
for i in range(n):           # O(n)
    x in window_set          # O(1)
    window_set.remove(x)     # O(1)
# total: O(n)


The general rule
Whenever you need fast membership checks (x in something),
use a set or dict — never a list.

A set is a hash map with no values, just keys.
x in set hashes x, checks that slot directly — O(1) always.

When a dict is better than a set
If you need to know how many times something appears in the window
(not just whether it's there), use a dict instead:


---
## Part 3 — Sliding Window vs Prefix Sum

Both are O(n) techniques for working with subarrays. Knowing when to use which is the skill.

| | Sliding Window | Prefix Sum |
|--|---------------|------------|
| **Window size** | fixed or variable | fixed |
| **Query type** | one pass, track best window | many arbitrary range queries |
| **Condition** | can shrink/expand based on rules | just sum any range |
| **Use when** | finding optimal subarray | answering multiple range queries |

**Rule of thumb:**
- Many range queries on same array → prefix sum
- Find the best/longest/shortest subarray satisfying a condition → sliding window

---
## Part 4 — Real World: Data Stream Applications

Sliding window is everywhere in data engineering and ML:

| Use Case | Window | Condition |
|----------|--------|----------|
| **Network monitoring** | last N packets | flag if sum > threshold |
| **Stock price** | last k days | find max/min price window |
| **NLP tokenization** | characters | longest valid token |
| **Fraud detection** | last N transactions | flag if any repeat merchant |
| **Log analysis** | time window | find busiest N-minute window |

In each case, sliding window lets you process a continuous stream of data in O(n) by maintaining a running window state instead of recomputing from scratch.

---
## LeetCode

Try for at least 20 minutes before asking for hints.

- [ ] MAXIMUM AVERAGE SUBARRAY I — fixed window
- [ ] Longest Substring Without Repeating Characters (Medium) — variable window

In [8]:
# MAXIMUM AVERAGE SUBARRAY I
# given an array nums and integer k, find the contiguous subarray of length k
# with the maximum average value, return that average
# example: nums=[1,12,-5,-6,50,3], k=4 → 12.75  (subarray [12,-5,-6,50], avg=51/4)
#
# this is a fixed window problem — same pattern as Part 1
# step 1: compute sum of first window
# step 2: slide — add nums[i], remove nums[i-k], update max sum
# step 3: return max_sum / k

def find_max_average(nums, k):
    window_sum = sum(nums[0:k])
    max_sum = window_sum

    for i in range(k, len(nums)):
        window_sum = window_sum + nums[i] - nums[i-k]
        if window_sum > max_sum:
            max_sum = window_sum
    return max_sum/k

print(find_max_average([1, 12, -5, -6, 50, 3], 4))  # 12.75
print(find_max_average([5], 1))                      # 5.0

12.75
5.0


In [7]:
# LONGEST SUBSTRING WITHOUT REPEATING CHARACTERS
# given a string s, find the length of the longest substring
# with no repeating characters
# example: "abcabcbb" → 3  ("abc")
# example: "bbbbb"    → 1  ("b")
# example: "pwwkew"   → 3  ("wke")
#
# this is the same pattern as Part 2 but on a string instead of an array
# left and right pointers, set to track characters in current window
# right moves forward each step
# when s[right] already in set → shrink from left until it's gone

def length_of_longest_substring(s):
    left = 0
    track_set = set()

    max_window = 0

    for i in range(len(s)):
        if s[i] not in track_set:
            track_set.add(s[i])
        else:
            # shrink from left until it's gone
            while s[i] in track_set:
                track_set.remove(s[left])
                left += 1
            track_set.add(s[i])
        max_window = max(max_window, len(track_set))
    return max_window
            

print(length_of_longest_substring("abcabcbb"))  # 3
print(length_of_longest_substring("bbbbb"))     # 1
print(length_of_longest_substring("pwwkew"))    # 3
print(length_of_longest_substring(""))          # 0

3
1
3
0


---
## Reflection

Write your answers in your own words:

1. What is the difference between fixed and variable size sliding window? When do you use each?  
=> In fixed sliding window the size of sliding window does not increase or decrease(it remains constant) while in variable size sliding window there are certain conditions depending on which the size of window keeps growing or shrinks.    
Fixed size is used when we want to find the max average, sum, least for a window size. And the variable one is used when we want to find the longest, biggest or something similar but with a condition.    
Fixed window when the problem gives us k, variable window when the problem asks us to find k.

2. Why is a set used instead of a list for tracking window contents in the variable window pattern?  
=>  The time complexity for lookup in set is O(1) but for a list it is O(n). Further if we use the set inside of a loop it becomes O(n) whereas for a list it becomes O(n**2). So this is the reason.  


3. When would you choose sliding window over prefix sums, and vice versa?  
=> In case of variable size of window or in case where we have to find the best/longest/optimal window then I will use sliding window. But for the cases where I have to find the aggregates or we have to run multiple queries I will use prefix sums.  

4. You're monitoring a live stream of user clicks. You want to flag any 10-minute window where the same user clicks more than 3 times. How would you use sliding window here?  
=> Here I will use a N-minute window , so the window size would be 10 and it will be for time in minutes. I will use a dictionary to maintain the record of user and their click counts. I will use that along with sliding window approach to flag the anomaly.    
The window is variable here — it contains all clicks within the last 10 minutes. As time moves forward, old clicks (older than 10 minutes) fall out from the left. The dict maps user_id → click count within the current window. When any user's count exceeds 3, flag it. When a click leaves the window, decrement that user's count in the dict — if it hits 0, remove them entirely.